In [2]:
import re

def parse(clause):
    return frozenset(clause.split(" | "))


def negate(literal):
    return literal[1:] if literal.startswith("~") else "~" + literal


def unify(a, b):
    a = re.findall(r"\w+", a)
    b = re.findall(r"\w+", b)

    if a[0] != b[0] or len(a) != len(b):
        return None

    sub = {}

    def replace(x):
        while x in sub:
            x = sub[x]
        return x

    for x, y in zip(a[1:], b[1:]):
        x, y = replace(x), replace(y)

        if x == y:
            continue
        elif x[0].islower():
            sub[x] = y
        elif y[0].islower():
            sub[y] = x
        else:
            return None

    return sub


def substitute(literal, sub):
    def replace(match):
        x = match.group()
        while x in sub:
            x = sub[x]
        return x

    return re.sub(r"\b\w+\b", replace, literal)


def resolve(c1, c2):
    for a in sorted(c1):
        for b in sorted(c2):

            if a.startswith("~") == b.startswith("~"):
                continue

            sub = unify(a, negate(b))

            if sub is not None:
                remaining = (c1 - {a}) | (c2 - {b})

                result = frozenset(
                    substitute(lit, sub)
                    for lit in remaining
                )

                yield result


def display(clause):
    return " | ".join(sorted(clause)) if clause else "{}"


def resolution(kb, query):
    clauses = [parse(c) for c in kb]
    goal = parse(negate(query))

    queue = [(goal, [])]
    visited = {goal}

    while queue:
        current, proof = queue.pop(0)

        for clause in clauses:
            for result in resolve(current, clause):

                if result in visited:
                    continue

                path = proof + [(current, clause, result)]

                if not result:
                    for i, (a, b, r) in enumerate(path, 1):
                        print(f"\nStep {i}:")
                        print("  ", display(a))
                        print("  ", display(b))
                        print("  =>", display(r))

                    return True

                visited.add(result)
                queue.append((result, path))

    return False


# Knowledge Base
KB = [
    "~Food(x) | Likes(John,x)",
    "Food(Apple)",
    "Food(Vegetables)",
    "~Eats(y,z) | Killed(y) | Food(z)",
    "Eats(Anil,Peanuts)",
    "Alive(Anil)",
    "~Eats(Anil,w) | Eats(Harry,w)",
    "Killed(g) | Alive(g)",
    "~Alive(k) | ~Killed(k)"
]

# Query
query = "Likes(John,Peanuts)"

print("===== FOL RESOLUTION =====")
print("\nKnowledge Base:")

for i, clause in enumerate(KB, 1):
    print(f"C{i}: {clause}")

print("\nQuery:", query)
print("Negated Query:", negate(query))
print("\n===== RESOLUTION STEPS =====")

if resolution(KB, query):
    print("\nContradiction Found: {}")
    print("Goal Proved:", query)
else:
    print("\nGoal Not Proved")

===== FOL RESOLUTION =====

Knowledge Base:
C1: ~Food(x) | Likes(John,x)
C2: Food(Apple)
C3: Food(Vegetables)
C4: ~Eats(y,z) | Killed(y) | Food(z)
C5: Eats(Anil,Peanuts)
C6: Alive(Anil)
C7: ~Eats(Anil,w) | Eats(Harry,w)
C8: Killed(g) | Alive(g)
C9: ~Alive(k) | ~Killed(k)

Query: Likes(John,Peanuts)
Negated Query: ~Likes(John,Peanuts)

===== RESOLUTION STEPS =====

Step 1:
   ~Likes(John,Peanuts)
   Likes(John,x) | ~Food(x)
  => ~Food(Peanuts)

Step 2:
   ~Food(Peanuts)
   Food(z) | Killed(y) | ~Eats(y,z)
  => Killed(y) | ~Eats(y,Peanuts)

Step 3:
   Killed(y) | ~Eats(y,Peanuts)
   Eats(Anil,Peanuts)
  => Killed(Anil)

Step 4:
   Killed(Anil)
   ~Alive(k) | ~Killed(k)
  => ~Alive(Anil)

Step 5:
   ~Alive(Anil)
   Alive(Anil)
  => {}

Contradiction Found: {}
Goal Proved: Likes(John,Peanuts)
